In [1]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

2026-01-30 19:59:59.496305: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 19:59:59.553014: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-30 20:00:01.464904: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-30 20:00:09.979016: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [3]:
layers = [4096, 2048, 1]
epochs = 50
act_func = tf.nn.relu
dropout = 0.5
input_dropout = 0.2
eta = 1e-5
norm = 'tanh'

In [ ]:
train_features, val_features, _, _, train_targets, val_targets, _, _ = load(norm=norm)
print("Training features shape:", train_features.shape)
print("Validation features shape:", val_features.shape)
print("Training targets shape:", train_targets.shape)
print("Validation targets shape:", val_targets.shape)

print("NaN in train_features:", np.isnan(train_features).any())
print("Inf in train_features:", np.isinf(train_features).any())
print("NaN in train_targets:", np.isnan(train_targets).any())
print("Inf in train_targets:", np.isinf(train_targets).any())

print("\nFirst 5 rows of train_features:\n", train_features[:5])
print("First 5 elements of train_targets:\n", train_targets[:5])


Training data shape: (13884, 7063)
Validation data shape: (4614, 7063)
Training targets shape: (13884, 1)
Validation targets shape: (4614, 1)
NaN in X_tr: False
Inf in X_tr: False
NaN in y_tr: False
Inf in y_tr: False

First 5 rows of X_tr:
 [[ 0.05802761 -0.84273285 -0.3678634  ...  0.          0.
   0.        ]
 [ 0.06155244 -0.84273285 -0.3678634  ... -0.50879097 -0.5930343
   0.07794365]
 [-0.30412385 -0.84273285 -0.3678634  ...  0.46145353  0.9597163
  -0.36973113]
 [-0.15402697 -0.84273285 -0.3678634  ...  0.          0.
   0.        ]
 [-0.4454553  -0.84273285 -0.3678634  ...  0.          0.
   0.        ]]
First 5 elements of y_tr:
 [[ 7.69353  ]
 [ 7.7780533]
 [-1.1985054]
 [ 2.5956845]
 [-5.1399713]]


In [ ]:
model = Sequential()
for i in range(len(layers)):
    if i == 0:
        model.add(Dense(
            layers[i],
            input_shape=(train_features.shape[1],),
            activation=act_func,
            kernel_initializer='he_normal'
        ))
        model.add(Dropout(float(input_dropout)))
    elif i == len(layers) - 1:
        model.add(Dense(
            layers[i],
            activation='linear',
            kernel_initializer="he_normal"
        ))
    else:
        model.add(Dense(
            layers[i],
            activation=act_func,
            kernel_initializer="he_normal"
        ))
        model.add(Dropout(float(dropout)))

 00:30:39.884718: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2112] Could not identify NUMA node of platform GPU id 0, defaulting to 0.  Your kernel may not have been built with NUMA support.
I0000 00:00:1769646639.884751    3567 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-29 00:30:39.884770: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 26831 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:02:00.0, compute capability: 12.0


In [ ]:
model.compile(
    loss='mean_squared_error',
    optimizer=K.optimizers.SGD(
        learning_rate=float(eta),
        momentum=0.5
    )
)
model.summary()

_________________________________________________________________


In [ ]:
history = model.fit(
    train_features, train_targets,
    epochs=epochs,
    batch_size=64,
    shuffle=True,
    validation_data=(val_features, val_targets),
    verbose=1
)

217/217 [==============================] - 1s 3ms/step - loss: 91.9796 - val_loss: 87.1200


In [ ]:
val_loss = history.history['val_loss']
train_loss = history.history['loss']
print("Final training loss:", train_loss[-1])
print("Final validation loss:", val_loss[-1])
model.reset_states()

Final training loss: 91.97958374023438
Final validation loss: 87.11997985839844
